In [1]:

import re, json
from pathlib import Path
import pandas as pd
import scipy.sparse
from sklearn.feature_extraction.text import CountVectorizer

DATASETS = Path("./datasets")
DATASETS.mkdir(parents=True, exist_ok=True)

def _limpar_texto(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r'[^a-zA-Z0-9áéíóúãõâêôçÁÉÍÓÚÃÕÂÊÔÇ\s]', ' ', s)
    s = re.sub(r'\b\d+\b', ' ', s)           # remove tokens puramente numéricos
    return re.sub(r'\s+', ' ', s).strip()

def transformar_phishing_jupyter(
    max_features=5000,
    min_df=5,
    max_df=0.8
) -> Path:
    """
    Lê phishing.csv (./datasets) e gera phishing_transformed.csv usando APENAS Bag of Words.
    Retorna o Path do CSV gerado.
    """
    # localizar phishing.csv
    in_candidates = [
        DATASETS / "phishing.csv",
        Path("../datasets/phishing.csv"),
        Path("phishing.csv"),
    ]
    src = next((p for p in in_candidates if p.exists()), None)
    if src is None:
        raise FileNotFoundError("Não encontrei 'phishing.csv'. Coloque-o em ./datasets/.")

    # leitura tolerante de encoding
    try:
        df = pd.read_csv(src, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(src, encoding="latin1")

    if "Email Text" not in df.columns:
        # já transformado? (colunas bow_*)
        bow_cols = [c for c in df.columns if c.startswith("bow_")]
        if not bow_cols:
            raise ValueError(
                "Coluna 'Email Text' não existe e não achei colunas 'bow_*'. "
                "Use o phishing.csv original."
            )

    # cria alvo se necessário
    if "Email Type_Phishing Email" not in df.columns:
        if "Email Type" in df.columns:
            df["Email Type_Phishing Email"] = df["Email Type"].astype(str).str.lower().str.contains("phishing").astype(int)
        else:
            df["Email Type_Phishing Email"] = 0  # fallback

    out_csv = DATASETS / "phishing_transformed.csv"
    out_npz = DATASETS / "phishing_bow_count.npz"
    out_vocab = DATASETS / "phishing_bow_vocab.json"

    # se tiver texto, gerar BoW
    if "Email Text" in df.columns:
        corpus = df["Email Text"].astype(str).apply(_limpar_texto)
        if corpus.dropna().str.strip().eq("").all():
            raise ValueError("Todos os textos ficaram vazios após a limpeza.")

        vectorizer = CountVectorizer(
            lowercase=True,
            stop_words=None,
            max_features=max_features,
            min_df=min_df,
            max_df=max_df,
        )
        X = vectorizer.fit_transform(corpus)
        print("BoW shape:", X.shape)

        # salvar forma esparsa e vocabulário (úteis p/ reprodução)
        scipy.sparse.save_npz(out_npz, X)
        with open(out_vocab, "w", encoding="utf-8") as f:
            json.dump(vectorizer.get_feature_names_out().tolist(), f, ensure_ascii=False, indent=2)

        bow_cols = [f"bow_{t}" for t in vectorizer.get_feature_names_out()]
        bow_df = pd.DataFrame.sparse.from_spmatrix(X, columns=bow_cols)

        base = df.drop(columns=["Email Text"], errors="ignore")
        out_df = pd.concat([base.reset_index(drop=True), bow_df.reset_index(drop=True)], axis=1)
    else:
        # já transformado com bow_*
        out_df = df.copy()

    out_df.to_csv(out_csv, index=False, encoding="utf-8")
    print("Gerado:", out_csv)
    return out_csv

def transformar_ddos_jupyter() -> Path | None:
    """
    Remove IPs/portas e salva ddos_transformed.csv.
    """
    candidates = [DATASETS/"DDoS.csv", Path("../datasets/DDoS.csv"), Path("DDoS.csv")]
    src = next((p for p in candidates if p.exists()), None)
    if src is None:
        print("[DDoS] CSV não encontrado — pulando.")
        return None

    try:
        df = pd.read_csv(src, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(src, encoding="latin1")

    for c in ["source IP","dest IP","source port","Dest Port"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    out = DATASETS / "ddos_transformed.csv"
    df.to_csv(out, index=False, encoding="utf-8")
    print("[DDoS] Gerado:", out)
    return out

def transformar_ransomware_jupyter() -> Path | None:
    """
    Remove colunas irrelevantes e salva ransomware_transformed.csv.
    """
    candidates = [DATASETS/"ransomware.csv", Path("../datasets/ransomware.csv"), Path("ransomware.csv")]
    src = next((p for p in candidates if p.exists()), None)
    if src is None:
        print("[Ransomware] CSV não encontrado — pulando.")
        return None

    try:
        df = pd.read_csv(src, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(src, encoding="latin1")

    for c in ["FileName", "md5Hash"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    out = DATASETS / "ransomware_transformed.csv"
    df.to_csv(out, index=False, encoding="utf-8")
    print("[Ransomware] Gerado:", out)
    return out


In [2]:

paths_gerados = {}

paths_gerados["ddos"] = transformar_ddos_jupyter()
paths_gerados["ransomware"] = transformar_ransomware_jupyter()

# phishing: se já existir o transformed, apenas informa; senão, gera com BoW
from pathlib import Path
phishing_out = Path("./datasets/phishing_transformed.csv")
if phishing_out.exists():
    print("[Phishing] Já existe phishing_transformed.csv — nada a fazer.")
    paths_gerados["phishing"] = phishing_out
else:
    paths_gerados["phishing"] = transformar_phishing_jupyter()

print("\n✅ Concluído. Arquivos transformados:")
for k, v in paths_gerados.items():
    print(f"  - {k}: {v}")


[DDoS] Gerado: datasets\ddos_transformed.csv
[Ransomware] Gerado: datasets\ransomware_transformed.csv
BoW shape: (18650, 5000)
Gerado: datasets\phishing_transformed.csv

✅ Concluído. Arquivos transformados:
  - ddos: datasets\ddos_transformed.csv
  - ransomware: datasets\ransomware_transformed.csv
  - phishing: datasets\phishing_transformed.csv
